In [ ]:
# For App GT Partner 
# Enhanced Eligibility Analysis - Continuous Forest & 2m Canopy
# Updated to accept polygon coordinates from app (lat/lng JSON format)

import geemap
import ee
import json
from datetime import datetime

# Initialize Earth Engine
ee.Initialize()

Map = geemap.Map()
Map.add_basemap('HYBRID')

Dynamic_World = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1')
scale = 10

# ============================================================
# NEW: Accept polygon from app as lat/lng JSON array
# ============================================================

def create_roi_from_coords(coords_json):
    """
    Convert app polygon format to Earth Engine FeatureCollection.
    
    Accepts either:
      - A JSON string: '[{"lat":24.30,"lng":82.82}, ...]'
      - A Python list:  [{"lat":24.30,"lng":82.82}, ...]
    
    Returns an ee.FeatureCollection (same as the old asset-based roi).
    """
    if isinstance(coords_json, str):
        coords = json.loads(coords_json)
    else:
        coords = coords_json

    # Earth Engine expects [lng, lat] order
    ee_coords = [[pt["lng"], pt["lat"]] for pt in coords]

    # Close the ring if not already closed
    if ee_coords[0] != ee_coords[-1]:
        ee_coords.append(ee_coords[0])

    geometry = ee.Geometry.Polygon([ee_coords])
    feature = ee.Feature(geometry)
    return ee.FeatureCollection([feature])


# ── Paste / inject your polygon here ──────────────────────────────────────────
polygon_from_app = [
    {"lat": 24.300745253658558, "lng": 82.82732952386141},
    {"lat": 24.301204218917405, "lng": 82.82751224935055},
    {"lat": 24.30096831953766,  "lng": 82.82815631479025},
    {"lat": 24.300406070543016, "lng": 82.82793637365103},
    {"lat": 24.300745253658558, "lng": 82.82732952386141},  # closing point (optional)
]
# ──────────────────────────────────────────────────────────────────────────────

roi = create_roi_from_coords(polygon_from_app)
Map.centerObject(roi)

# ============================================================
# Everything below is unchanged from your original script
# ============================================================

# Define analysis period (2016-2024)
analysis_years = list(range(2016, 2025))

def get_yearly_landcover(year):
    """Get annual land cover classification"""
    start_date = f'{year}-01-01'
    end_date = f'{year}-12-31'
    yearly_data = Dynamic_World.filterBounds(roi) \
        .filterDate(start_date, end_date) \
        .select('label') \
        .mode()
    return yearly_data

# Generate yearly land cover
print("Processing land cover data 2016-2024...")
yearly_landcover = {}
for year in analysis_years:
    yearly_landcover[year] = get_yearly_landcover(year)

# Create forest masks for each year
forest_masks = {}
for year in analysis_years:
    forest_masks[year] = yearly_landcover[year].eq(1)  # Trees class

# Detect CONTINUOUSLY FORESTED areas
print("Identifying continuously forested areas...")

def get_continuous_forest_mask(min_years=3):
    total_forest_years = forest_masks[2016]
    for year in range(2017, 2025):
        total_forest_years = total_forest_years.add(forest_masks[year])
    return total_forest_years.gte(min_years)

continuous_forest = get_continuous_forest_mask(min_years=3)
was_ever_continuous_forest = continuous_forest

current_2024 = yearly_landcover[2024]
non_eligible_current = current_2024.eq(1)

eligible_areas = was_ever_continuous_forest.Not().And(non_eligible_current.Not())

# Canopy Height Exclusion - 2m threshold
canopy_height = ee.ImageCollection(
    'projects/meta-forest-monitoring-okw37/assets/CanopyHeight'
).mosaic().clip(roi)
canopy_exclusion = canopy_height.gte(2)

final_eligible = eligible_areas.And(canopy_exclusion.Not())

# Convert to vectors
def clip_eligible_to_polygon(feature):
    geometry = feature.geometry()
    poly_eligible_mask = final_eligible.clip(geometry)
    eligible_vectors = poly_eligible_mask.selfMask().reduceToVectors(
        geometry=geometry,
        scale=scale,
        geometryType='polygon',
        crs='EPSG:4326',
        bestEffort=True,
        maxPixels=1e13
    )
    eligible_with_props = eligible_vectors.map(
        lambda f: f.set(feature.toDictionary())
    )
    return eligible_with_props

all_eligible_parts = roi.map(clip_eligible_to_polygon).flatten()

# Visualization
vis_params_dw = {
    'bands': ['label'],
    'palette': ['#419BDF','#397D49','#88B053','#7A87C6','#E49635',
                '#DFC35A','#C4281B','#A59B8F','#B39FE1'],
    'min': 0, 'max': 8
}

Map.addLayer(yearly_landcover[2016].clip(roi), vis_params_dw, '2016 Baseline')
Map.addLayer(yearly_landcover[2024].clip(roi), vis_params_dw, '2024 Current')
Map.addLayer(continuous_forest.selfMask().clip(roi), {'palette': ['red']}, 'Continuous Forest (3+ years)', False)
Map.addLayer(canopy_height, {'min': 0, 'max': 20, 'palette': ['white','green']}, 'Canopy Height (m)', False)
Map.addLayer(final_eligible.selfMask().clip(roi), {'palette': ['00FF00']}, 'Eligible Areas (Raster)')
Map.addLayer(all_eligible_parts, {'color': '00FF00', 'fillColor': '00FF0044'}, 'Eligible Areas (Vector)')

# Area Calculations
print("=== AREA ANALYSIS ===")

total_area = roi.geometry().area().divide(10000)
total_ha = total_area.getInfo()
print(f"Total area: {total_ha:.2f} hectares")

print("\nForest area by year:")
for year in analysis_years:
    forest_area = forest_masks[year].multiply(ee.Image.pixelArea()) \
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=scale).get('label')
    forest_ha = ee.Number(forest_area).divide(10000).getInfo()
    print(f"  {year}: {forest_ha:.2f} ha")

continuous_forest_ha = ee.Number(
    continuous_forest.multiply(ee.Image.pixelArea())
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=scale).get('label')
).divide(10000).getInfo()

current_forest_ha = ee.Number(
    non_eligible_current.multiply(ee.Image.pixelArea())
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=scale).get('label')
).divide(10000).getInfo()

canopy_excl_ha = ee.Number(
    canopy_exclusion.multiply(ee.Image.pixelArea())
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=scale).get('cover_code')
).divide(10000).getInfo()

final_eligible_ha = ee.Number(
    final_eligible.multiply(ee.Image.pixelArea())
    .reduceRegion(reducer=ee.Reducer.sum(), geometry=roi, scale=scale).get('label')
).divide(10000).getInfo()

print(f"\n=== EXCLUSION SUMMARY ===")
print(f"Total area:                          {total_ha:.2f} ha")
print(f"Excluded - Continuous forest (3+ yr): {continuous_forest_ha:.2f} ha")
print(f"Excluded - Current forest (2024):     {current_forest_ha:.2f} ha")
print(f"Excluded - Canopy height >=2m:        {canopy_excl_ha:.2f} ha")
print(f"\n{'='*60}")
print(f"FINAL ELIGIBLE AREA FOR A/R PROJECT: {final_eligible_ha:.2f} HECTARES")
print(f"{'='*60}")
print(f"\n=== BREAKDOWN ===")
print(f"Total area:    {total_ha:.2f} ha")
print(f"Total excluded: {(total_ha - final_eligible_ha):.2f} ha")
print(f"Eligible area: {final_eligible_ha:.2f} ha ({final_eligible_ha/total_ha*100:.1f}% of total)")

print(f"\n=== METHODOLOGY ===")
print("✓ Excludes areas that were forest in 3+ years (avoiding 1-year anomalies)")
print("✓ Excludes areas that are currently forest (2024)")
print("✓ Excludes areas with canopy height >=2m (existing trees)")
print("✓ ROI loaded from app polygon (lat/lng JSON) — no GEE asset required")

# Export
export_task = ee.batch.Export.table.toDrive(
    collection=all_eligible_parts,
    description='lot3appsbd3',
    folder='SBDapp1_1',
    fileFormat='KML'
)
export_task.start()
print(f"\nExport started: eligible_areas.kml")

Map.addLayerControl()
Map